In [3]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pywt
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import os
import h5py
from temporaldata import Data

import mne

from mne.time_frequency import tfr_array_morlet

from auditorydecoding.windowing import extract_windows
from auditorydecoding.plotting import plot_signal, plot_pca_variance, plot_covariance, plot_spectrum
from scipy.signal import butter, sosfilt

In [5]:
DATA_ROOT = "/capstor/scratch/cscs/awgolab/foundry_test/data/processed/neurosoft_monkeys_2026"
RECORDING_ID = "sub-01_ses-01_task-AcousStim_acq-RH_desc-raw"
S_FREQ = 2000

In [6]:
path = os.path.join(DATA_ROOT, RECORDING_ID + ".h5")

with h5py.File(path) as f:
    data = Data.from_hdf5(f, lazy=False)

CHANNEL_NAMES = data.channels.id

In [ ]:
keep_channels = data.channels.type == "ecog"
channel_names = data.channels.id[keep_channels]

In [ ]:
ecog_data = data.ecog.signal[:,keep_channels].T

info = mne.create_info(ch_names=list(data.channels.id[keep_channels]), sfreq=S_FREQ, ch_types=list(data.channels.type[keep_channels]))
raw = mne.io.RawArray(ecog_data, info)

raw.notch_filter(freqs=50)

onset_samples   =   (data.on_vs_off_trials.start * S_FREQ).astype(int)
event_ids       =   data.on_vs_off_trials.behavior_ids

events = np.column_stack([
	onset_samples,
	np.zeros(len(onset_samples), dtype=int),
	event_ids
])

event_id = dict(zip(data.on_vs_off_trials.behavior_labels, data.on_vs_off_trials.behavior_ids))

epochs = mne.Epochs(
    raw, events,
    event_id=event_id,        # pass the dict — lets you select by label later
    tmin=-0.1,
    tmax=0.5,
    baseline=(-0.1, 0),
    preload=True
)

epochs.plot(n_epochs=10, n_channels=10)


### Preprocessing

In [7]:
def preprocess_raw(raw, notch_freqs=(50.0,), notch_widths=None, verbose=False):
    """
    Apply notch filter(s) to remove line noise from a Raw object.

    Parameters
    ----------
    raw          : mne.io.RawArray  -- modified in-place AND returned
    notch_freqs  : tuple of float   -- fundamental(s) to notch, e.g. (50,) or (60,)
                   Harmonics up to Nyquist are added automatically.
    notch_widths : float or None    -- bandwidth per notch (MNE default: 1 Hz)
    verbose      : bool

    Returns
    -------
    raw : mne.io.RawArray  (filtered in-place)
    """
    sfreq   = raw.info["sfreq"]
    nyquist = sfreq / 2.0

    freqs_to_notch = []
    for fund in notch_freqs:
        harmonic = fund
        while harmonic < nyquist:
            freqs_to_notch.append(harmonic)
            harmonic += fund

    if not freqs_to_notch:
        return raw

    kwargs = dict(freqs=freqs_to_notch, verbose=verbose)
    if notch_widths is not None:
        kwargs["notch_widths"] = notch_widths

    raw.notch_filter(**kwargs)
    if verbose:
        print(f"  Notch-filtered at: {freqs_to_notch} Hz")
    return raw

In [ ]:
def make_epochs(raw, trial_starts, trial_stops, trial_ids,
                baseline=None, verbose=False):
    """
    Build an mne.Epochs object from sample-indexed trial boundaries.

    Parameters
    ----------
    raw          : mne.io.RawArray
    trial_starts : np.ndarray (n_trials,)  -- sample indices
    trial_stops  : np.ndarray (n_trials,)  -- sample indices
    trial_ids    : array-like (n_trials,)  -- condition labels (str or numeric)
    baseline     : tuple or None           -- e.g. (None, 0) for pre-stim baseline
    verbose      : bool

    Returns
    -------
    epochs : mne.Epochs
    """
    sfreq       = raw.info["sfreq"]
    labels      = np.asarray(trial_ids)
    unique_lbls = np.unique(labels)

    onset_samples   =   (data.on_vs_off_trials.start * S_FREQ).astype(int)
    event_ids       =   data.on_vs_off_trials.behavior_ids

    events = np.column_stack([
        onset_samples,
        np.zeros(len(onset_samples), dtype=int),
        event_ids
    ])

    event_id = dict(zip(data.on_vs_off_trials.behavior_labels, data.on_vs_off_trials.behavior_ids))

    epochs = mne.Epochs(
        raw, events,
        event_id=event_id,        # pass the dict — lets you select by label later
        tmin=-0.1,
        tmax=0.5,
        baseline=(-0.1, 0),
        preload=True
    )

    # MNE event IDs must be positive integers
    label_to_id = {lbl: i + 1 for i, lbl in enumerate(unique_lbls)}
    id_to_label = {v: k for k, v in label_to_id.items()}

    # Build events array: (n_events, 3) = [sample, 0, event_id]
    events = np.column_stack([
        trial_starts.astype(int),
        np.zeros(len(trial_starts), dtype=int),
        np.array([label_to_id[lbl] for lbl in labels], dtype=int),
    ])

    # Epoch duration: use the median trial length so all epochs are equal-length
    durations_s  = (trial_stops - trial_starts) / sfreq
    median_dur_s = float(np.median(durations_s))
    tmin, tmax   = 0.0, median_dur_s

    event_id_dict = {str(lbl): label_to_id[lbl] for lbl in unique_lbls}

    epochs = mne.Epochs(
        raw, events, event_id=event_id_dict,
        tmin=tmin, tmax=tmax,
        baseline=baseline,
        preload=True,
        verbose=verbose,
    )

    # Store reverse mapping so downstream code can recover original labels
    epochs._label_map = id_to_label
    return epochs


def get_epoch_labels(epochs):
    """
    Recover original string/numeric labels in epoch order.
    """
    label_map = epochs._label_map
    return np.array([label_map[ev] for ev in epochs.events[:, 2]])